# 02 — BRD4780 walkthrough: MUC1 → TMED9

**Disease:** Autosomal dominant tubulointerstitial kidney disease, MUC1 type (ADTKD-MUC1)  
**Causal gene:** `MUC1` (also `UMOD` for the related ADTKD-UMOD form) — a frameshift produces a toxic, misfolded protein (MUC1-fs) that accumulates intracellularly  
**Preclinically validated strategy (gold):** Block `TMED9` cargo-receptor function to release MUC1-fs from the early secretory pathway and direct it to lysosomal degradation  
**Tool compound:** BRD4780 (Greka lab, Broad Institute)

This is the harder case. The intervention target (`TMED9`) is **not** the disease gene, not in the disease gene's primary signaling pathway, and only emerges when you understand that the disease is a protein-trafficking failure rather than a kidney-function failure.

## 1. Load the case

In [ ]:
from fda_strategy_triples import load_cases

case = next(c for c in load_cases() if c.case_id == 'adtkd-muc1')
print(case.disease_gene, '->', case.gold_triple.intervention_target)
print('Mechanism:', case.gold_triple.mechanism)

## 2. Retrieve pathway context for MUC1

In [ ]:
from g2p_rag import Retriever

retriever = Retriever.from_pretrained('v0.1.0')
context = retriever.retrieve(case.disease_gene, k=10)

for snippet in context[:5]:
    print(f"[{snippet.source}] {snippet.text[:120]}...")

What we want to see in the retrieved context: snippets mentioning ER-to-Golgi cargo trafficking, p24 family proteins, and unfolded protein response, in addition to canonical MUC1 mucin biology. If only the latter shows up, the agent has no chance.

## 3. Run the agent

In [ ]:
from therapy_agent import TherapyAgent

agent = TherapyAgent(model='claude-opus-4-7')
hypotheses = agent.propose(case.disease_gene, context)

for i, h in enumerate(hypotheses, 1):
    print(f"{i}. target={h.target} modality={h.modality} conf={h.confidence:.2f}")
    print(f"   rationale: {h.rationale[:200]}...")

## 4. Score

In [ ]:
from bio_rag_eval import score_case

score = score_case(case, hypotheses)
print(f"Recovered: {score.recovered}")
print(f"Rank of correct target: {score.rank}")
print(f"Judge score: {score.judge_score:.2f}")
print(f"Judge rationale: {score.judge_rationale}")

## Why this case matters

If you only follow the disease gene's first-degree neighbors, the answer (TMED9) is invisible. The reasoning has to chain: MUC1 frameshift → misfolded protein → trapped in early secretory pathway → which cargo receptor holds it there → can we release it. A retrieval system that returns only "things known to bind MUC1" is useless here; one that returns p24-family / TMED snippets when asked about misfolded membrane proteins is what makes the agent succeed.

This is the test case that distinguishes a real reasoning agent from a lookup table.